# Load packages

In [1]:
import scimap as sm

import pandas as pd
from sklearn.neighbors import BallTree
import numpy as np
from joblib import Parallel, delayed
import scipy
from functools import reduce
from scipy.spatial import Delaunay

import seaborn as sns
import matplotlib.pyplot as plt

Running SCIMAP  2.2.11


# Load data

In [2]:
image_path = '/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Data_AllMarkers_filtered_RCNs.csv'
# Convert the data to scimap format
adata = sm.pp.mcmicro_to_scimap(image_path, 
                                remove_dna=False, remove_string_from_name=None, log=False, drop_markers=None,
                                random_sample=None, unique_CellId=True, CellId='CellID', split='X_centroid',
                                custom_imageid=None, min_cells=None, output_dir=None)
adata.obsm['spatial'] = np.array(adata.obs[['Y_centroid', 'X_centroid']])
adata

Loading Data_AllMarkers_filtered_RCNs.csv


/Users/wenqchen/miniconda/envs/napari-env/lib/python3.10/site-packages/scimap/preprocessing/mcmicro_to_scimap.py:106: DtypeWarning:

Columns (33) have mixed types. Specify dtype option on import or set low_memory=False.



AnnData object with n_obs × n_vars = 3681772 × 25
    obs: 'X_centroid', 'Y_centroid', 'core_imageid', 'cores.x', 'celltype', 'Annotation', 'Stage', 'patient_id_AB19_1654', 'Area', 'Eccentricity', 'antioxidant_NQOI', 'antioxidant_GCLC', 'antioxidant_TXNRD1', 'antioxidant_GLUT1', 'NQO1_GCLC_VIM', 'celltype_2', 'celltype_1', 'Tumor_state', 'GCLC_VIM', 'CellCategory', 'SC45px_neigh_kmeans_10', 'RCNs', 'CellID', 'imageid'
    uns: 'all_markers'
    obsm: 'spatial'

In [3]:
adata.obs

,X_centroid,Y_centroid,core_imageid,cores.x,celltype,Annotation,Stage,patient_id_AB19_1654,Area,Eccentricity,...,NQO1_GCLC_VIM,celltype_2,celltype_1,Tumor_state,GCLC_VIM,CellCategory,SC45px_neigh_kmeans_10,RCNs,CellID,imageid
19_02_02A_1,2027.642,1011.2380,core77_19_02_02A,core77,Other,Invasive border,2,AB19-1654_P10158,282,0.821700,...,---,Other,Other.Stroma,Other.Stroma,Other.Stroma,Stroma,4,RCN3,1,19_02_02A
19_02_01A_10,5871.136,777.1124,core77_19_02_01A,core77,Tumor,Tumor center,2,AB19-1654_P10117,258,0.620470,...,+--,Tumor_+--,Tumor,Tumor,Tumor.GCLC-VIM-,Tumor,2,RCN1,10,19_02_01A
19_02_02A_10,1999.424,1195.5040,core77_19_02_02A,core77,CD68.Macrophages,Invasive border,2,AB19-1654_P10158,125,0.755401,...,+--,CD68.Macrophages,CD68.Macrophages,CD68.Macrophages,CD68.Macrophages,Immune,4,RCN3,10,19_02_02A
19_02_01A_100,6136.928,858.5450,core77_19_02_01A,core77,Tumor,Tumor center,2,AB19-1654_P10117,222,0.803584,...,+--,Tumor_+--,Tumor,Tumor,Tumor.GCLC-VIM-,Tumor,2,RCN1,100,19_02_01A
19_02_02A_100,1728.540,1302.0830,core77_19_02_02A,core77,Immune,Invasive border,2,AB19-1654_P10158,300,0.522613,...,---,Immune,Other.Immune,Other.Immune,Other.Immune,Immune,4,RCN3,100,19_02_02A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
INF1_99998,36889.410,6446.4120,core22_INF1,core22,Tumor,Inflammation,unknown,AB19-1654_P10648,148,0.641780,...,---,Tumor_---,Tumor,Tumor,Tumor.GCLC-VIM-,Tumor,2,RCN1,99998,INF1
1B_99999,11181.530,7239.2680,core64_1B,core64,FOXP3.CD4.Tregs,Tumor center,2,AB19-1654_P10561,366,0.771436,...,-++,FOXP3.CD4.Tregs,Treg,Treg,Treg,Immune,1,RCN2,99999,1B
2B_99999,8371.249,6579.8230,core71_2B,core71,CD4.T.cells,Lymph node met,2,AB19-1654_P10599,558,0.863484,...,-++,CD4.T.cells,CD4.T.cells,CD4.T.cells,CD4.T.cells,Immune,1,RCN2,99999,2B
3B_99999,20828.410,4477.0180,core42_3B,core42,Tumor,Tumor center,1,AB19-1654_P10620,112,0.883622,...,---,Tumor_---,Tumor,Tumor,Tumor.GCLC-VIM-,Tumor,2,RCN1,99999,3B


# Function

## COZI function

In [4]:

# Function
def spatial_interaction (adata,
                         x_coordinate='X_centroid',
                         y_coordinate='Y_centroid',
                         z_coordinate=None,
                         phenotype='phenotype',
                         method='radius',
                         radius=30,
                         knn=10,
                         permutation=1000,
                         cond_counts_threshold=5,
                         imageid='imageid',
                         subset=None,
                         pval_method='zscore',
                         normalization='total',
                         verbose=True,
                         scaling=False,
                         label='spatial_interaction',
                         perm_dir=None):
    """
Parameters:
        adata (anndata.AnnData):  
            Annotated data matrix or path to an AnnData object, containing spatial gene expression data.

        x_coordinate (str, required):  
            Column name in `adata` for the x-coordinates.

        y_coordinate (str, required):  
            Column name in `adata` for the y-coordinates.

        z_coordinate (str, optional):  
            Column name in `adata` for the z-coordinates, for 3D spatial data analysis.

        phenotype (str, required):  
            Column name in `adata` indicating cell phenotype or any categorical cell classification.

        method (str, optional):  
            Method to define neighborhoods: 'radius' for fixed distance, 'knn' for K nearest neighbors and 'delaunay' for Delaunay triangulation.

        radius (int, optional):  
            Radius for neighborhood definition (applies when method='radius').

        knn (int, optional):  
            Number of nearest neighbors to consider (applies when method='knn').

        permutation (int, optional):  
            Number of permutations for p-value calculation.

        cond_counts_threshold (int, optional):
            Minimum number of observed conditional interactions required for a cell type pair to be considered.
            Pairs with conditional counts below this threshold will be set to 0, only applied when normalization = 'conditional'. Default is 5.

        imageid (str, required):  
            Column name in `adata` for image identifiers, useful for analysis within specific images.

        subset (str, optional):  
            Specific image identifier for targeted analysis.

        pval_method (str, optional):  
            Method for p-value calculation: 'abs' for absolute difference, 'zscore' for z-score based significance.

        normalization (str, optional):
            Method for normalization: 'total' for total cell count normalization, 'conditional' for conditional normalization (adapted from histocat).
        
        verbose (bool):  
            If set to `True`, the function will print detailed messages about its progress and the steps being executed.

        label (str, optional):  
            Custom label for storing results in `adata.obs`.

Returns:
        adata (anndata.AnnData):  
            Updated `adata` object with spatial interaction results in `adata.obs[label]`.

Example:
        ```python
        
        # Radius method for 2D data with absolute p-value calculation
        adata = sm.tl.spatial_interaction(adata, x_coordinate='X_centroid', y_coordinate='Y_centroid',
                                    method='radius', radius=50, permutation=1000, pval_method='abs',
                                    label='interaction_radius_abs')
    
        # KNN method for 2D data with z-score based p-value calculation
        adata = sm.tl.spatial_interaction(adata, x_coordinate='X_centroid', y_coordinate='Y_centroid',
                                    method='knn', knn=15, permutation=1000, pval_method='zscore',
                                    label='interaction_knn_zscore')
    
        # Radius method for 3D data analysis
        adata = sm.tl.spatial_interaction(adata, x_coordinate='X_centroid', y_coordinate='Y_centroid',
                                    z_coordinate='Z_centroid', method='radius', radius=60, permutation=1000,
                                    pval_method='zscore', label='interaction_3D_zscore')
        
        ```
    """
    
    
    def spatial_interaction_internal (adata_subset,
                                      x_coordinate,
                                      y_coordinate,
                                      z_coordinate,
                                      phenotype,
                                      method,
                                      radius,
                                      knn,
                                      permutation, 
                                      imageid,
                                      subset,
                                      pval_method,
                                      normalization):
        if verbose:
            print("Processing Image: " + str(adata_subset.obs[imageid].unique()))
        
        # Create a dataFrame with the necessary information
        # This is useful for 3D data or 2D data with Z coordinate (multi stacks)
        if z_coordinate is not None:
            if verbose:
                print("Including Z -axis")
            data = pd.DataFrame({'x': adata_subset.obs[x_coordinate], 'y': adata_subset.obs[y_coordinate], 'z': adata_subset.obs[z_coordinate], 'phenotype': adata_subset.obs[phenotype]})
        else:
            data = pd.DataFrame({'x': adata_subset.obs[x_coordinate], 'y': adata_subset.obs[y_coordinate], 'phenotype': adata_subset.obs[phenotype]})

        
        # Select the neighborhood method, knn, radius or delaunay
        # a) KNN method
        if method == 'knn':
            if verbose:
                print("Identifying the " + str(knn) + " nearest neighbours for every cell")
            if z_coordinate is not None:
                tree = BallTree(data[['x','y','z']], leaf_size= 2)
                ind = tree.query(data[['x','y','z']], k=knn, return_distance= False)
            else:
                tree = BallTree(data[['x','y']], leaf_size= 2)
                ind = tree.query(data[['x','y']], k=knn, return_distance= False)
            neighbours = pd.DataFrame(ind.tolist(), index = data.index) # neighbour DF
            neighbours.drop(0, axis=1, inplace=True) # Remove self neighbour
            
        # b) Local radius method
        if method == 'radius':
            if verbose:
                print("Identifying neighbours within " + str(radius) + " pixels of every cell")
            if z_coordinate is not None:
                kdt = BallTree(data[['x','y','z']], metric='euclidean') 
                ind = kdt.query_radius(data[['x','y','z']], r=radius, return_distance=False)
            else:
                kdt = BallTree(data[['x','y']], metric='euclidean') 
                ind = kdt.query_radius(data[['x','y']], r=radius, return_distance=False)
                
            for i in range(0, len(ind)): ind[i] = np.delete(ind[i], np.argwhere(ind[i] == i))#remove self
            neighbours = pd.DataFrame(ind.tolist(), index = data.index) # neighborhood DF

        # c) Delaunay triangulation method
        if method == 'delaunay':
            if verbose:
                print("Performing Delaunay triangulation to identify neighbours for every cell")
            if z_coordinate is not None:
                points = data[['x', 'y', 'z']].values
            else:
                points = data[['x', 'y']].values

            # Perform Delaunay triangulation
            delaunay = Delaunay(points)

            # Initialize a dictionary to store neighbours
            neighbours_dict = {i: set() for i in range(len(points))}

            # Iterate over each simplex (triangle/tetrahedron) to populate the neighbours dictionary
            for simplex in delaunay.simplices:
                for i in range(len(simplex)):
                    for j in range(i + 1, len(simplex)):
                        neighbours_dict[simplex[i]].add(simplex[j])
                        neighbours_dict[simplex[j]].add(simplex[i])

            # Convert the neighbours dictionary to a list of lists
            neighbours_list = [list(neighbours) for neighbours in neighbours_dict.values()]

            # Ensure each list has the same number of elements by padding with -1 (assuming indices are non-negative)
            max_neigh_len = max(len(neigh) for neigh in neighbours_list)
            neighbours_list_padded = [neigh + [-1] * (max_neigh_len - len(neigh)) for neigh in neighbours_list]

            # Convert to numpy array for consistency with KNN method
            ind = np.array(neighbours_list_padded)

            # Convert to DataFrame for the same output format as the original function
            neighbours = pd.DataFrame(ind.tolist(), index=data.index)

            # Replace -1 with None
            neighbours.replace(-1, None, inplace=True)

        ### END OF NEIGHBORHOOD SELECTION ###
        # Map Phenotypes to Neighbours
        # Loop through (all functionized methods were very slow)
        phenomap = dict(zip(list(range(len(ind))), data['phenotype'])) # Used for mapping
        if verbose:
            print("Mapping phenotype to neighbors")
        for i in neighbours.columns:
            neighbours[i] = neighbours[i].dropna().map(phenomap, na_action='ignore')
            
        # Drop NA
        neighbours = neighbours.dropna(how='all')
        
        # Collapse all the neighbours into a single column
        n = pd.DataFrame(neighbours.stack(), columns = ["neighbour_phenotype"])
        n.index = n.index.get_level_values(0) # Drop the multi index
        
        # Merge with real phenotype
        n = n.merge(data['phenotype'], how='inner', left_index=True, right_index=True)
        
        # Permutation
        if verbose:
            print('Performing '+ str(permutation) + ' permutations')

        #### Permutation ####
        # Set a global seed for reproducibility
        np.random.seed(42)

        # Generate fixed seeds for all permutations
        seeds = np.random.randint(0, 1e6, size=permutation) 

        def permutation_pval (data, seed):
           # Permute the neighbour_phenotype column without affecting the original data structure
            # set seed
            np.random.seed(seed)
            data = data.assign(neighbour_phenotype=np.random.permutation(data['neighbour_phenotype']))
            k = data.groupby(['phenotype','neighbour_phenotype'],observed=False).size().unstack()#.fillna(0)
            
            # add neighbour phenotype that are not present to make k a square matrix
            columns_to_add = dict.fromkeys(np.setdiff1d(k.index,k.columns), 0)
            k = k.assign(**columns_to_add)
            total_cell_count = data.reset_index().drop_duplicates(subset=['index', 'phenotype']).groupby('phenotype').size().reindex(k.index, fill_value=0)  # Ensure all categories are included
            data_freq = k.div(total_cell_count, axis = 0)
            data_freq = data_freq.fillna(0).stack().values  # Flatten the matrix
            return data_freq

        def permutation_pval_norm (data, seed):
            # Permute the neighbour_phenotype column without affecting the original data structure
            # set seed
            np.random.seed(seed)
            data = data.assign(neighbour_phenotype=np.random.permutation(data['neighbour_phenotype']))
            data_freq = data.groupby(['phenotype','neighbour_phenotype'],observed=False).size().unstack()

            # Remove duplicate interactions (conditional factor)
            data = data.reset_index()
            data = data.drop_duplicates()
            data = data.set_index('index')

            # We noralize the data based on the number of cells of each type 
            normalization_factor = data.groupby(['phenotype', 'neighbour_phenotype'],observed=False).size().unstack()
            data_freq = data_freq/normalization_factor
            data_freq = data_freq.fillna(0).stack().values
            return data_freq
        
        # Apply permutation functions depending on normalization
        if normalization == "total":
            final_scores = Parallel(n_jobs=-1)(
                delayed(permutation_pval)(data=n, seed=seeds[i]) for i in range(permutation))
        if normalization == "conditional":
            final_scores = Parallel(n_jobs=-1)(
                delayed(permutation_pval_norm)(data=n, seed=seeds[i]) for i in range(permutation))

        # Permutation results
        perm = pd.DataFrame(final_scores).T
        
        # Include the image ID in the filename
        current_image_id = str(adata_subset.obs[imageid].unique()[0])
        perm.to_csv(f'{perm_dir}/permutation_{current_image_id}.csv')
        #perm.to_csv(f'/Users/wenqchen/Desktop/Projects/Kras/Data/COZI_perm/permutation_{current_image_id}.csv')
        #print("perm:", perm)
        
        # Consolidate the permutation results
        if verbose:
            print('Consolidating the permutation results')

        # Calculate P value
        # N_freq is the observed frequency of each cell type with each of its neighbours (observed number of interactions)
        if normalization == "total":
            # Calculate interaction frequencies without dropping any categories
            # Normalize based on total cell count
            k = n.groupby(['phenotype','neighbour_phenotype'],observed=False).size().unstack()#.fillna(0)
            # add neighbour phenotype that are not present to make k a square matrix
            columns_to_add = dict.fromkeys(np.setdiff1d(k.index,k.columns), 0)
            k = k.assign(**columns_to_add)
            #total_cell_count = data['phenotype'].value_counts().reindex(k.columns, fillvalue=0).values
            #total_cell_count = data.reset_index().drop_duplicates(subset=['index', 'phenotype']).groupby('phenotype').size().reindex(k.index, fill_value=0)  # Ensure all categories are included
            total_cell_count = data['phenotype'].value_counts()
            n_freq = k.div(total_cell_count, axis = 0)
            n_freq = n_freq.fillna(0).stack()  # Flatten the matrix

        # Normalize n_freq if normalization is conditional
        if normalization == "conditional":
            # Calculate observed interaction frequencies
            data = n.assign(neighbour_phenotype=n['neighbour_phenotype'])
            data_freq = n.groupby(['phenotype', 'neighbour_phenotype'], observed=False).size().unstack()

            # Remove duplicate interactions (conditional factor)
            data = data.reset_index()
            data = data.drop_duplicates()
            data = data.set_index('index')

            normalization_factor = data.groupby(['phenotype', 'neighbour_phenotype'],observed=False).size().unstack()

            # Calculate percentage of pairs below threshold for warning
            below_threshold = (normalization_factor < cond_counts_threshold).sum().sum()
            total_pairs = normalization_factor.size
            perc_below = (below_threshold / total_pairs) * 100
            
            if perc_below > 0 and verbose:
                print(f"Warning: {perc_below:.1f}% of cell type pairs have counts below {cond_counts_threshold}. "
                      "Results for these pairs should be interpreted with caution.")
            
            mask = normalization_factor < cond_counts_threshold
            data_freq = data_freq / normalization_factor
            data_freq[mask] = np.nan
            n_freq = data_freq.fillna(0).stack()
   
        # permutation with scaling
        if scaling == True:
            perm_scaled = perm.apply(lambda row: 2 * (row - row.min()) / (row.max() - row.min()) - 1, axis=1)
            mean = perm_scaled.mean(axis=1)
            std = perm_scaled.std(axis=1)
            # Initialize a new Series to store scaled `n_freq`
            n_freq_scaled = n_freq.copy()

            # Normalize `n_freq` using the min and max of the corresponding rows in `perm`
            for i in range(len(n_freq_scaled)):
                row_min = perm.iloc[i, :].min()
                row_max = perm.iloc[i, :].max()
                n_freq_scaled.iloc[i] = 2 * (n_freq.iloc[i] - row_min) / (row_max - row_min) - 1
                n_freq = n_freq_scaled
        else:
            mean = perm.mean(axis=1)
            std = perm.std(axis=1)

        # P-value calculation
        if pval_method == 'abs':
            # Calculate the number of times permuted values exceed the observed
            p_values = np.sum(perm >= n_freq.values[:, None], axis=1) / (permutation + 1)
            p_values = p_values[~np.isnan(p_values)].values

        if pval_method == 'zscore':
            z_scores = (n_freq.values - mean) / std        
            z_scores[np.isnan(z_scores)] = 0
            p_values = scipy.stats.norm.sf(abs(z_scores))*2
            p_values = p_values[~np.isnan(p_values)]

        # Compute Direction of interaction (interaction or avoidance)
        direction = ((n_freq.values - mean) / abs(n_freq.values - mean)).fillna(1)


        # DataFrame with the neighbour frequency and P values
        if pval_method == 'abs':
            count = (n_freq.values * direction).values # adding directionallity to interaction
            neighbours = pd.DataFrame({'count': count, 'p_val': p_values}, index=n_freq.index)
            neighbours.columns = [adata_subset.obs[imageid].unique()[0],
                                  'pvalue_' + str(adata_subset.obs[imageid].unique()[0])]
            neighbours = neighbours.reset_index()

        elif pval_method == 'zscore':
            #count = (n_freq.values * direction).values # adding directionality to interaction
            count = n_freq.values
            neighbours = pd.DataFrame({'z_score':z_scores.values,'p_val': p_values, 'count':n_freq}, index = n_freq.index)
            neighbours.columns = ['zscore_' + str(adata_subset.obs[imageid].unique()[0]),
                                  'pvalue_' + str(adata_subset.obs[imageid].unique()[0]),
                                  'count_' + str(adata_subset.obs[imageid].unique()[0])]
            neighbours = neighbours.reset_index()
        
        # Return the results
        return neighbours
          
      
    # subset a particular subset of cells if the user wants else break the adata into list of anndata objects
    if subset is not None:
        adata_list = [adata[adata.obs[imageid] == subset]]
    else:
        adata_list = [adata[adata.obs[imageid] == i] for i in adata.obs[imageid].unique()]
    
    
    # Apply function to all images and create a master dataframe
    # Create lamda function 
    r_spatial_interaction_internal = lambda x: spatial_interaction_internal (adata_subset=x,
                                                                             x_coordinate=x_coordinate,
                                                                             y_coordinate=y_coordinate,
                                                                             z_coordinate=z_coordinate,
                                                                             phenotype=phenotype,
                                                                             method=method,
                                                                             radius=radius,
                                                                             knn=knn,
                                                                             permutation=permutation,
                                                                             imageid=imageid,
                                                                             subset=subset,
                                                                             pval_method=pval_method,
                                                                             normalization=normalization)

    # Apply function to all images
    all_data = list(map(r_spatial_interaction_internal, adata_list)) # Apply function

    # Merge all the results into a single dataframe    
    df_merged = reduce(lambda  left,right: pd.merge(left,right,on=['phenotype', 'neighbour_phenotype'], how='outer'), all_data)

    # Add to anndata
    adata.uns[label] = df_merged
    
    # return
    return adata

## Get neighbors

In [13]:
def get_neighbors(  adata,
                    x_coordinate,
                    y_coordinate,
                    z_coordinate=None,
                    phenotype='phenotype',
                    method='knn',
                    radius=50,
                    knn=10,
                    imageid='imageid',
                    subset=None,
                    output_csv=None,
                    verbose=True):
    """
    Get neighbors of each cell in the dataset.
    """
    def get_neighbors_internal (adata_subset,x_coordinate,y_coordinate,z_coordinate,phenotype,method,radius,knn,
                                imageid,subset):
        
        if verbose:
            print("Processing Image: " + str(adata_subset.obs[imageid].unique()))
        # Create a dataFrame with the necessary information
        # This is useful for 3D data or 2D data with Z coordinate (multi stacks)
        if z_coordinate is not None:
            if verbose:
                print("Including Z -axis")
            data = pd.DataFrame({'x': adata_subset.obs[x_coordinate], 'y': adata_subset.obs[y_coordinate], 'z': adata_subset.obs[z_coordinate], 'phenotype': adata_subset.obs[phenotype]})
        else:
            data = pd.DataFrame({'x': adata_subset.obs[x_coordinate], 'y': adata_subset.obs[y_coordinate], 'phenotype': adata_subset.obs[phenotype]})
            
        # Select the neighborhood method, knn, radius or delaunay
        # a) KNN method
        if method == 'knn':
            if verbose:
                print("Identifying the " + str(knn) + " nearest neighbours for every cell")
            if z_coordinate is not None:
                tree = BallTree(data[['x','y','z']], leaf_size= 2)
                ind = tree.query(data[['x','y','z']], k=knn, return_distance= False)
            else:
                tree = BallTree(data[['x','y']], leaf_size= 2)
                ind = tree.query(data[['x','y']], k=knn, return_distance= False)
            neighbours = pd.DataFrame(ind.tolist(), index = data.index) # neighbour DF
            neighbours.drop(0, axis=1, inplace=True) # Remove self neighbour
            
        # b) Local radius method
        if method == 'radius':
            if verbose:
                print("Identifying neighbours within " + str(radius) + " pixels of every cell")
            if z_coordinate is not None:
                kdt = BallTree(data[['x','y','z']], metric='euclidean') 
                ind = kdt.query_radius(data[['x','y','z']], r=radius, return_distance=False)
            else:
                kdt = BallTree(data[['x','y']], metric='euclidean') 
                ind = kdt.query_radius(data[['x','y']], r=radius, return_distance=False)
                
            for i in range(0, len(ind)): ind[i] = np.delete(ind[i], np.argwhere(ind[i] == i))#remove self
            neighbours = pd.DataFrame(ind.tolist(), index = data.index) # neighborhood DF
        
        # c) Delaunay triangulation method
        if method == 'delaunay':
            if verbose:
                print("Performing Delaunay triangulation to identify neighbours for every cell")
            if z_coordinate is not None:
                points = data[['x', 'y', 'z']].values
            else:
                points = data[['x', 'y']].values

            # Perform Delaunay triangulation
            delaunay = Delaunay(points)

            # Initialize a dictionary to store neighbours
            neighbours_dict = {i: set() for i in range(len(points))}

            # Iterate over each simplex (triangle/tetrahedron) to populate the neighbours dictionary
            for simplex in delaunay.simplices:
                for i in range(len(simplex)):
                    for j in range(i + 1, len(simplex)):
                        neighbours_dict[simplex[i]].add(simplex[j])
                        neighbours_dict[simplex[j]].add(simplex[i])

            # Convert the neighbours dictionary to a list of lists
            neighbours_list = [list(neighbours) for neighbours in neighbours_dict.values()]

            # Ensure each list has the same number of elements by padding with -1 (assuming indices are non-negative)
            max_neigh_len = max(len(neigh) for neigh in neighbours_list)
            neighbours_list_padded = [neigh + [-1] * (max_neigh_len - len(neigh)) for neigh in neighbours_list]

            # Convert to numpy array for consistency with KNN method
            ind = np.array(neighbours_list_padded)

            # Convert to DataFrame for the same output format as the original function
            neighbours = pd.DataFrame(ind.tolist(), index=data.index)

            # Replace -1 with None
            neighbours.replace(-1, None, inplace=True)
            
        ### END OF NEIGHBORHOOD SELECTION ###
        # Map Phenotypes to Neighbours
        # Loop through (all functionized methods were very slow)
        # Map phenotype
        phenomap = dict(zip(list(range(len(ind))), data['phenotype'])) # Used for mapping

        # Loop through (all functionized methods were very slow)
        for i in neighbours.columns:
            neighbours[i] = neighbours[i].dropna().map(phenomap, na_action='ignore')

        # Drop NA
        n_dropped = neighbours.dropna(how='all')

        # Collapse all the neighbours into a single column
        n = pd.DataFrame(neighbours.stack(), columns = ["neighbour_phenotype"])
        n.index = n.index.get_level_values(0) # Drop the multi index
        n = pd.DataFrame(n)
        n['order'] = list(range(len(n)))

        # Merge with real phenotype
        n_m = n.merge(data['phenotype'], how='inner', left_index=True, right_index=True)
        n_m['neighbourhood'] = n_m.index
        n = n_m.sort_values(by=['order'])
        
        n['samples'] = adata_subset.obs[imageid]

        return n
    
    # Subset a particular image if needed
    if subset is not None:
        adata_list = [adata[adata.obs[imageid] == subset]]
    else:
        adata_list = [adata[adata.obs[imageid] == i] for i in adata.obs[imageid].unique()]

    # Apply function to all images and create a master dataframe
    # Create lamda function 
    r_get_neighbors_internal = lambda x: get_neighbors_internal(adata_subset=x,x_coordinate=x_coordinate,
                                                   y_coordinate=y_coordinate,
                                                   z_coordinate=z_coordinate,
                                                   phenotype=phenotype,
                                                   method=method,radius=radius,knn=knn,
                                                   imageid=imageid,subset=subset) 
    all_data = list(map(r_get_neighbors_internal, adata_list)) # Apply function 


    # Merge all the results into a single dataframe    
    result = []
    for i in range(len(all_data)):
        result.append(all_data[i])
    result = pd.concat(result, join='outer')  
    
    if output_csv is not None:
        result.to_csv(output_csv, index=False)
        if verbose:
            print("Results saved to " + output_csv)
            
    # Return        
    return result

# Run

## COZI -- All data

In [ ]:
adata = spatial_interaction(adata,
                           x_coordinate='X_centroid',
                           y_coordinate='Y_centroid',
                           phenotype='GCLC_VIM',
                           method='radius',
                           radius=45,
                           #knn=10,
                           permutation=1000,
                           cond_counts_threshold=5,
                           imageid='core_imageid',
                           subset=None,
                           pval_method='zscore',
                           normalization='conditional',
                           verbose=True,
                           scaling=False,
                           label='COZI_radius_45px',
                           perm_dir='/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/COZI/COZI_perm'
                           )
#adata.write('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/COZI/All_COZI_45px.h5ad')

In [ ]:
COZI_data = adata.uns['COZI_radius_45px']
COZI_data.to_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/COZI/COZI_radius_45px.csv')

In [25]:
COZI_data['phenotype'].unique()

array(['CD206.Macrophages', 'CD4.T.cells', 'CD68.Macrophages',
       'CD8.T.cells', 'Other.Immune', 'Other.Stroma', 'SMA+.Stroma',
       'Treg', 'Tumor.GCLC+VIM+', 'Tumor.GCLC+VIM-', 'Tumor.GCLC-VIM+',
       'Tumor.GCLC-VIM-', 'VIM+.Stroma'], dtype=object)

## Get neighbors

In [14]:
result = get_neighbors(adata,
                        x_coordinate='X_centroid',
                        y_coordinate='Y_centroid',
                        z_coordinate=None,
                        phenotype='GCLC_VIM',
                        method='radius',
                        radius=45,
                        knn=None,
                        imageid='core_imageid',
                        subset=None,
                        output_csv='/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/Auria_phenotype_neighbors_45px.csv',
                        verbose=True)

Processing Image: ['core77_19_02_02A']
Categories (1, object): ['core77_19_02_02A']
Identifying neighbours within 45 pixels of every cell
Processing Image: ['core77_19_02_01A']
Categories (1, object): ['core77_19_02_01A']
Identifying neighbours within 45 pixels of every cell
Processing Image: ['core42_T2']
Categories (1, object): ['core42_T2']
Identifying neighbours within 45 pixels of every cell
Processing Image: ['core42_INF2']
Categories (1, object): ['core42_INF2']
Identifying neighbours within 45 pixels of every cell
Processing Image: ['core70_19_02_01A']
Categories (1, object): ['core70_19_02_01A']
Identifying neighbours within 45 pixels of every cell
Processing Image: ['core14_19_02_03A']
Categories (1, object): ['core14_19_02_03A']
Identifying neighbours within 45 pixels of every cell
Processing Image: ['core63_1B']
Categories (1, object): ['core63_1B']
Identifying neighbours within 45 pixels of every cell
Processing Image: ['core77_T2']
Categories (1, object): ['core77_T2']
Id

## Normalization factor

In [15]:
sample_list = result['samples'].unique()
print(f"Processing {len(sample_list)} samples...")

# Create an empty list to collect low_norm_factors for each sample
all_low_norm_factors = []
sample_stats = []

# Loop through each sample
for i, sample_name in enumerate(sample_list):
    print(f"Processing sample {i+1}/{len(sample_list)}: {sample_name}")
    
    try:
        # Subset data for the current sample
        result_subset = result[result['samples'] == sample_name]
        
        # Calculate observed interaction frequencies
        data = result_subset.assign(neighbour_phenotype=result_subset['neighbour_phenotype'])
        data_freq = result_subset.groupby(['phenotype', 'neighbour_phenotype'], observed=False).size().unstack()
        
        # Remove duplicate interactions (conditional factor)
        date_filter = data.copy()
        date_filter = date_filter.drop_duplicates()
        normalization_factor = date_filter.groupby(['phenotype', 'neighbour_phenotype'], observed=False).size().unstack()
        
        # Find phenotype-neighbour combinations with normalization factor < 5
        low_norm_factors = normalization_factor.stack().reset_index()
        low_norm_factors.columns = ['phenotype', 'neighbour_phenotype', 'normalization_factor']
        low_norm_factors = low_norm_factors[low_norm_factors['normalization_factor'] < 5]
        low_norm_factors = low_norm_factors.sort_values('normalization_factor')
        low_norm_factors['samples'] = sample_name
        
        # Calculate percentage of pairs below threshold for stats
        below_threshold = (normalization_factor < 5).sum().sum()
        total_pairs = normalization_factor.size
        perc_below = (below_threshold / total_pairs) * 100
        
        # Add stats to the collection
        sample_stats.append({
            'sample': sample_name,
            'total_pairs': total_pairs,
            'below_threshold': below_threshold,
            'percent_below': perc_below
        })
        
        # Add to the collection
        all_low_norm_factors.append(low_norm_factors)
        
    except Exception as e:
        print(f"Error processing sample {sample_name}: {str(e)}")

# Combine all low_norm_factors into a single dataframe
combined_low_norm_factors = pd.concat(all_low_norm_factors, ignore_index=True)

# Create a dataframe with sample statistics
sample_stats_df = pd.DataFrame(sample_stats)

print(f"Completed processing {len(sample_list)} samples.")
print(f"Total low normalization factor pairs: {len(combined_low_norm_factors)}")


Processing 345 samples...
Processing sample 1/345: core77_19_02_02A
Processing sample 2/345: core77_19_02_01A
Processing sample 3/345: core42_T2
Processing sample 4/345: core42_INF2
Processing sample 5/345: core70_19_02_01A
Processing sample 6/345: core14_19_02_03A
Processing sample 7/345: core63_1B
Processing sample 8/345: core77_T2
Processing sample 9/345: core64_1B
Processing sample 10/345: core71_2B
Processing sample 11/345: core42_3B
Processing sample 12/345: core22_INF1
Processing sample 13/345: core22_19_02_03A
Processing sample 14/345: core15_19_02_03A
Processing sample 15/345: core7_19_02_03A
Processing sample 16/345: core8_19_02_03A
Processing sample 17/345: core21_19_02_03A
Processing sample 18/345: core1_19_02_03A
Processing sample 19/345: core29_T2
Processing sample 20/345: core15_INF1
Processing sample 21/345: core56_3B
Processing sample 22/345: core16_T2
Processing sample 23/345: core57_19_02_01A
Processing sample 24/345: core1_INF1
Processing sample 25/345: core28_3B
Pr

In [17]:
combined_low_norm_factors.to_csv('/Users/wenqchen/Desktop/Projects/Auria/Data/t-CycIF/COZI/COZI_low_norm_factors.csv', index=False)

In [22]:
combined_low_norm_factors

,phenotype,neighbour_phenotype,normalization_factor,samples
0,CD206.Macrophages,SMA+.Stroma,1.0,core77_19_02_02A
1,CD206.Macrophages,Tumor.GCLC-VIM+,1.0,core77_19_02_02A
2,SMA+.Stroma,CD206.Macrophages,1.0,core77_19_02_02A
3,Tumor.GCLC-VIM+,CD206.Macrophages,1.0,core77_19_02_02A
4,CD8.T.cells,SMA+.Stroma,3.0,core77_19_02_02A
...,...,...,...,...
5013,Other.Immune,Treg,4.0,core34_3B
5014,Other.Immune,Tumor.GCLC-VIM+,4.0,core34_3B
5015,Other.Stroma,Other.Immune,4.0,core34_3B
5016,Treg,Other.Immune,4.0,core34_3B


In [26]:
df = combined_low_norm_factors[combined_low_norm_factors['phenotype'].isin(['Tumor.GCLC+VIM+', 'Tumor.GCLC-VIM+', 'Tumor.GCLC+VIM-', 'Tumor.GCLC-VIM-'])].copy()

In [27]:
df

,phenotype,neighbour_phenotype,normalization_factor,samples
3,Tumor.GCLC-VIM+,CD206.Macrophages,1.0,core77_19_02_02A
8,Tumor.GCLC+VIM+,SMA+.Stroma,3.0,core77_19_02_02A
9,Tumor.GCLC-VIM+,Treg,3.0,core77_19_02_02A
11,Tumor.GCLC+VIM-,Treg,1.0,core77_19_02_01A
13,Tumor.GCLC+VIM-,Tumor.GCLC-VIM+,1.0,core77_19_02_01A
...,...,...,...,...
5005,Tumor.GCLC-VIM-,Other.Immune,2.0,core34_3B
5009,Tumor.GCLC+VIM+,Treg,3.0,core34_3B
5010,Tumor.GCLC-VIM+,CD206.Macrophages,3.0,core34_3B
5011,Tumor.GCLC-VIM-,Treg,3.0,core34_3B


In [23]:
df = combined_low_norm_factors[combined_low_norm_factors['phenotype'] =='Tumor.GCLC+VIM+'].copy()
df

,phenotype,neighbour_phenotype,normalization_factor,samples
8,Tumor.GCLC+VIM+,SMA+.Stroma,3.0,core77_19_02_02A
32,Tumor.GCLC+VIM+,Tumor.GCLC+VIM-,2.0,core77_19_02_01A
53,Tumor.GCLC+VIM+,Treg,1.0,core42_T2
61,Tumor.GCLC+VIM+,CD206.Macrophages,2.0,core42_T2
73,Tumor.GCLC+VIM+,Other.Immune,3.0,core42_T2
...,...,...,...,...
4947,Tumor.GCLC+VIM+,Other.Stroma,2.0,core69_3B
4968,Tumor.GCLC+VIM+,Tumor.GCLC+VIM-,3.0,core62_3B
4971,Tumor.GCLC+VIM+,CD8.T.cells,4.0,core62_3B
5003,Tumor.GCLC+VIM+,Other.Stroma,4.0,core41_3B
